In [1]:
import pandas as pd
df = pd.read_csv('Dataset/dataset.csv')
df.describe()


,text,dial
count,477693,477693
unique,424930,12
top,"A questo titolo corrispondono più, di seguito ...",it
freq,18819,256065


In [ ]:
print(df.shape)
print(df.columns)
print(df.index)

(477693, 2)
Index(['text', 'dial'], dtype='str')
RangeIndex(start=0, stop=477693, step=1)


In [11]:
display(df['dial'].unique())
counts = df['dial'].value_counts()
percentages = df['dial'].value_counts(normalize=True) * 100
stats_df = pd.DataFrame({"N_Frasi": counts, "Percentuale (%)": percentages.round(2)})
display(stats_df)

<StringArray>
[     'eml',      'fur',      'lij',      'lld',      'lmo',      'nap',
      'pms', 'roa_tara',       'sc',      'scn',      'vec',       'it']
Length: 12, dtype: str

,N_Frasi,Percentuale (%)
dial,,
it,256065,53.60
vec,68497,14.34
lmo,50008,10.47
scn,25317,5.30
eml,12990,2.72
lld,11960,2.50
nap,11701,2.45
lij,10901,2.28
pms,9842,2.06


## Step 1: Pulizia Markup/Tag HTML, Sentence Splitting con Tracking Documentale (`doc_id`)

In questa fase:
1. **Rilevamento e conteggio**: identifichiamo e quantifichiamo tag HTML (`<br>`, `<ref>`, ecc.), entità HTML (`&amp;`, `&quot;`, ecc.) e intestazioni di sezione (`== Sezione ==`).
2. **Pulizia e normalizzazione**: rimuoviamo markup e tag residui, decodifichiamo le entità HTML e normalizziamo gli spazi.
3. **Sentence Splitting con `doc_id`**: dividiamo ciascun articolo in frasi tracciando l'ID dell'articolo originale (`doc_id`), prerequisito indispensabile per effettuare lo split a livello di documento ed evitare il data leakage.
4. **Filtraggio di qualità**: scartiamo frammenti troppo brevi (< 40 caratteri) e frasi di disambiguazione boilerplate.

In [17]:
import re
import html
import time
import pandas as pd

# ---------------------------------------------------------------------------
# 1. RILEVAMENTO E CONTEGGIO TAG HTML, ENTITÀ E MARKUP RESIDUO
# ---------------------------------------------------------------------------
html_tag_re = re.compile(r"<[^>]+>")
html_entity_re = re.compile(r"&[a-zA-Z]+;|&#\d+;")
wiki_header_re = re.compile(r"={2,}[^=\n]+={2,}")
wiki_template_re = re.compile(r"\{\{[^}]*\}\}")
whitespace_re = re.compile(r"\s+")
# Sentence boundary: divide su . ! ? seguito da spazio e lettera maiuscola / virgolette
sent_splitter_re = re.compile(r"(?<=[.!?])\s+(?=[A-ZÀ-ÖØ-ß\"«\d])")

print("=== ANALISI MARKUP RESIDUO SUL DATASET ORIGINALE ===")
has_html_tags = df['text'].str.contains(html_tag_re, regex=True, na=False)
has_html_entities = df['text'].str.contains(html_entity_re, regex=True, na=False)
has_wiki_headers = df['text'].str.contains(wiki_header_re, regex=True, na=False)
has_templates = df['text'].str.contains(wiki_template_re, regex=True, na=False)
has_any_markup = has_html_tags | has_html_entities | has_wiki_headers | has_templates

total_docs = len(df)
print(f"Totale documenti iniziali: {total_docs:,}")
print(f"- Con tag HTML (<...>):             {has_html_tags.sum():8,} ({has_html_tags.mean():.2%})")
print(f"- Con entità HTML (&amp;, etc.):    {has_html_entities.sum():8,} ({has_html_entities.mean():.2%})")
print(f"- Con intestazioni Wiki (==...==):  {has_wiki_headers.sum():8,} ({has_wiki_headers.mean():.2%})")
print(f"- Con template residui ({{...}}):   {has_templates.sum():8,} ({has_templates.mean():.2%})")
print(f"-> Osservazioni totali con markup:  {has_any_markup.sum():8,} ({has_any_markup.mean():.2%})")

# ---------------------------------------------------------------------------
# 2. PULIZIA TESTI, SENTENCE SPLITTING CON TRACKING DOC_ID E FILTRI
# ---------------------------------------------------------------------------
def clean_and_normalize(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = html_tag_re.sub(" ", text)
    text = wiki_header_re.sub(" ", text)
    text = wiki_template_re.sub(" ", text)
    return whitespace_re.sub(" ", text).strip()

print("\nAvvio pulizia, sentence splitting e tracciamento doc_id...")
t0 = time.time()

MIN_CHAR_LEN = 40
sentences = []

# Enumeriamo per mantenere doc_id univoco per ciascun articolo
for doc_id, (raw_text, dial) in enumerate(zip(df['text'], df['dial'])):
    clean_doc = clean_and_normalize(raw_text)
    if not clean_doc or len(clean_doc) < MIN_CHAR_LEN:
        continue
    
    doc_sents = sent_splitter_re.split(clean_doc)
    for sent in doc_sents:
        sent = sent.strip()
        if len(sent) >= MIN_CHAR_LEN and not sent.startswith("A questo titolo corrispondono"):
            sentences.append((doc_id, sent, dial))

df_sents = pd.DataFrame(sentences, columns=["doc_id", "text", "dial"])
raw_sentences_count = len(df_sents)
t_split = time.time() - t0
print(f"-> Estratte {raw_sentences_count:,} frasi in {t_split:.1f} secondi.")

# Deduplicazione frasi esatte
df_sents.drop_duplicates(subset=["text"], inplace=True)
print(f"-> Frasi uniche totali: {len(df_sents):,}")


=== ANALISI MARKUP RESIDUO SUL DATASET ORIGINALE ===
Totale documenti iniziali: 477,693
- Con tag HTML (<...>):                    0 (0.00%)
- Con entità HTML (&amp;, etc.):      27,520 (5.76%)
- Con intestazioni Wiki (==...==):        13 (0.00%)
- Con template residui ({...}):        591 (0.12%)
-> Osservazioni totali con markup:    28,074 (5.88%)

Avvio pulizia, sentence splitting e tracciamento doc_id...
-> Estratte 6,144,993 frasi in 113.6 secondi.
-> Frasi uniche totali: 5,833,379


In [18]:
# Salvataggio del dataset di frasi pulite completo con doc_id
print("Salvataggio in corso di 'Dataset/dataset_clean_sentences.csv'...")
df_sents.to_csv('Dataset/dataset_clean_sentences.csv', index=False)
print(f"-> Dataset pulito salvato con successo! ({len(df_sents):,} frasi, colonne: {list(df_sents.columns)})")


Salvataggio in corso di 'Dataset/dataset_clean_sentences.csv'...
-> Dataset pulito salvato con successo! (5,833,379 frasi, colonne: ['doc_id', 'text', 'dial'])


## Step 2: Stratified Document-Level Split (70% Train, 15% Validation, 15% Test)

### Perché lo split a livello di documento è fondamentale?
Se dividessimo casualmente le frasi, frasi provenienti dallo stesso articolo di Wikipedia (es. *"Storia di Venezia"*) finirebbero sia nel Train che nel Test. Il modello finirebbe per memorizzare i **topic/entità** (nomi propri, eventi storici) anziché imparare le reali caratteristiche sintattiche e morfologiche del dialetto (**topic leakage / data contamination**).

### Metodologia:
1. Identifichiamo gli ID unici degli articoli (`doc_id`) per ciascun dialetto.
2. Eseguiamo uno split stratificato **70% Train, 15% Validation, 15% Test** direttamente sui `doc_id`.
3. Assegniamo le frasi ai rispettivi split in base al loro `doc_id` di appartenenza.
4. Rimuoviamo da Validation e Test eventuali frasi duplicate generiche presenti nel Train, garantendo che i dati di valutazione siano al **100% inediti**.

In [19]:
from sklearn.model_selection import train_test_split

print("=== ESECUZIONE STRATIFIED DOCUMENT-LEVEL SPLIT (70 / 15 / 15) ===")

# Se df_sents non è già presente in memoria, lo carichiamo dal CSV appena salvato
if 'df_sents' not in locals():
    print("Caricamento di df_sents da 'Dataset/dataset_clean_sentences.csv'...")
    df_sents = pd.read_csv('Dataset/dataset_clean_sentences.csv')

# Controllo di integrità: verifica che la colonna doc_id sia presente
if 'doc_id' not in df_sents.columns:
    raise ValueError("La colonna 'doc_id' non è presente in df_sents. Assicurati di eseguire prima la cella dello Step 1 e la cella di salvataggio!")

# 1. Estrazione degli articoli unici con la rispettiva classe dialettale
unique_docs = df_sents[['doc_id', 'dial']].drop_duplicates()
print(f"Articoli unici totali da splittare: {len(unique_docs):,}")

# 2. Primo split stratificato: 70% Train, 30% Temporaneo (per Val e Test)
train_docs, temp_docs = train_test_split(
    unique_docs,
    test_size=0.30,
    random_state=42,
    stratify=unique_docs['dial']
)

# 3. Secondo split stratificato: divide il 30% a metà (15% Val, 15% Test)
val_docs, test_docs = train_test_split(
    temp_docs,
    test_size=0.50,
    random_state=42,
    stratify=temp_docs['dial']
)

print(f"- Articoli Train:      {len(train_docs):8,} ({len(train_docs)/len(unique_docs):.1%})")
print(f"- Articoli Validation: {len(val_docs):8,} ({len(val_docs)/len(unique_docs):.1%})")
print(f"- Articoli Test:       {len(test_docs):8,} ({len(test_docs)/len(unique_docs):.1%})")

# 4. Mappatura delle frasi sui rispettivi split documentali
train_df = df_sents[df_sents['doc_id'].isin(train_docs['doc_id'])].copy()
val_df = df_sents[df_sents['doc_id'].isin(val_docs['doc_id'])].copy()
test_df = df_sents[df_sents['doc_id'].isin(test_docs['doc_id'])].copy()

# 5. Anti-leakage: rimuoviamo da Val e Test eventuali frasi identiche già viste in Train
train_text_set = set(train_df['text'])
val_df = val_df[~val_df['text'].isin(train_text_set)].copy()
test_df = test_df[~test_df['text'].isin(train_text_set)].copy()

# 6. Verifica formale dell'assenza di Data Leakage
doc_leak_train_val = len(set(train_df['doc_id']).intersection(set(val_df['doc_id'])))
doc_leak_train_test = len(set(train_df['doc_id']).intersection(set(test_df['doc_id'])))
text_leak_train_test = len(set(train_df['text']).intersection(set(test_df['text'])))

print("\n--- VERIFICA INTEGRITÀ & ZERO DATA LEAKAGE ---")
print(f"Overlap articoli Train <-> Val:  {doc_leak_train_val} (atteso: 0)")
print(f"Overlap articoli Train <-> Test: {doc_leak_train_test} (atteso: 0)")
print(f"Overlap frasi Train <-> Test:    {text_leak_train_test} (atteso: 0)")
assert doc_leak_train_val == 0 and doc_leak_train_test == 0 and text_leak_train_test == 0, "Errore di leakage rilevato!"
print("-> Convalida superata: nessun leakage tra gli insiemi!")

# 7. Statistiche finali delle frasi per dialetto su Train, Val e Test
print("\n=== CONTEGGIO FRASI PER DIALETTO NEI TRE SPLIT ===")
summary_table = pd.DataFrame({
    'Train': train_df['dial'].value_counts(),
    'Val': val_df['dial'].value_counts(),
    'Test': test_df['dial'].value_counts(),
    'Totale': df_sents['dial'].value_counts()
}).fillna(0).astype(int)

summary_table['Train %'] = (summary_table['Train'] / summary_table['Totale'] * 100).round(1)
summary_table['Val %'] = (summary_table['Val'] / summary_table['Totale'] * 100).round(1)
summary_table['Test %'] = (summary_table['Test'] / summary_table['Totale'] * 100).round(1)

display(summary_table)
print(f"\nTotale frasi: Train={len(train_df):,}, Val={len(val_df):,}, Test={len(test_df):,}")


=== ESECUZIONE STRATIFIED DOCUMENT-LEVEL SPLIT (70 / 15 / 15) ===
Articoli unici totali da splittare: 418,299
- Articoli Train:       292,809 (70.0%)
- Articoli Validation:   62,745 (15.0%)
- Articoli Test:         62,745 (15.0%)

--- VERIFICA INTEGRITÀ & ZERO DATA LEAKAGE ---
Overlap articoli Train <-> Val:  0 (atteso: 0)
Overlap articoli Train <-> Test: 0 (atteso: 0)
Overlap frasi Train <-> Test:    0 (atteso: 0)
-> Convalida superata: nessun leakage tra gli insiemi!

=== CONTEGGIO FRASI PER DIALETTO NEI TRE SPLIT ===


,Train,Val,Test,Totale,Train %,Val %,Test %
dial,,,,,,,
it,3660795,783487,788044,5232326,70.0,15.0,15.1
lmo,125129,25987,26163,177279,70.6,14.7,14.8
vec,85609,18767,17858,122234,70.0,15.4,14.6
scn,57636,11448,11434,80518,71.6,14.2,14.2
sc,40704,8738,8153,57595,70.7,15.2,14.2
lij,28047,6100,6805,40952,68.5,14.9,16.6
lld,20634,4466,4249,29349,70.3,15.2,14.5
roa_tara,16762,3956,4088,24806,67.6,15.9,16.5
nap,16283,3804,3812,23899,68.1,15.9,16.0



Totale frasi: Train=4,082,970, Val=873,361, Test=877,048


In [20]:
# Salvataggio dei 3 split su disco (pronti per essere usati da qualsiasi modello)
print("Salvataggio split in corso in Dataset/...")
train_df[['text', 'dial']].to_csv('Dataset/train_sentences.csv', index=False)
val_df[['text', 'dial']].to_csv('Dataset/val_sentences.csv', index=False)
test_df[['text', 'dial']].to_csv('Dataset/test_sentences.csv', index=False)
print("-> Split salvati con successo: train_sentences.csv, val_sentences.csv, test_sentences.csv!")


Salvataggio split in corso in Dataset/...
-> Split salvati con successo: train_sentences.csv, val_sentences.csv, test_sentences.csv!


## Step 3: Feature Extraction (TF-IDF), Scaling Sparso e Baseline Logistic Regression (SAG)

In questo step replichiamo la **Baseline 1** del paper *VarDial 2022* (*"The Curious Case of Logistic Regression for Italian Languages and Dialects Identification"*):

1. **TfidfVectorizer**: estraiamo le feature testuali (unigrammi e bigrammi). Per evitare data leakage, il vocabolario viene calcolato con `fit_transform` **esclusivamente sul Train set** e applicato con `transform` sul Validation set.
2. **StandardScaler su matrice sparsa**: usiamo `StandardScaler(with_mean=False)`. È fondamentale impostare `with_mean=False` per non de-sparsificare la matrice TF-IDF; sottrarre la media trasformerebbe milioni di zeri in numeri densi, causando un crash istantaneo della memoria RAM (OOM).
3. **Logistic Regression con solver `sag` e `class_weight='balanced'`**: addestriamo il classificatore lineare multi-classe con l'algoritmo Stochastic Average Gradient e la compensazione dello sbilanciamento delle classi.
4. **Valutazione metrica**: calcoliamo **Macro F1-score**, Weighted F1-score e la matrice di confusione.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay

print("=== AVVIO STEP 3: BASELINE LINEARE TF-IDF + LOGISTIC REGRESSION ===\n")

# ---------------------------------------------------------------------------
# 1. TF-IDF VECTORIZATION (FIT ESCLUSIVO SUL TRAIN)
# ---------------------------------------------------------------------------
t0 = time.time()
print("1. Estrazione feature TF-IDF in corso...")

# Usiamo unigrammi e bigrammi con un cap sulle feature più informative
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True,
    min_df=3
)

X_train_tfidf = tfidf.fit_transform(train_df['text'])
X_val_tfidf = tfidf.transform(val_df['text'])

y_train = train_df['dial'].values
y_val = val_df['dial'].values

print(f"-> TF-IDF completato in {time.time()-t0:.1f}s")
print(f"-> Matrice Train TF-IDF: {X_train_tfidf.shape[0]:,} campioni x {X_train_tfidf.shape[1]:,} feature (nnz: {X_train_tfidf.nnz:,})")
print(f"-> Matrice Val TF-IDF:   {X_val_tfidf.shape[0]:,} campioni x {X_val_tfidf.shape[1]:,} feature\n")

# ---------------------------------------------------------------------------
# 2. STANDARD SCALING SPARSE-SAFE (with_mean=False)
# ---------------------------------------------------------------------------
t0_scale = time.time()
print("2. Scaling della varianza con StandardScaler(with_mean=False)...")

# NOTA: with_mean=False scala solo la deviazione standard preservando la sparsity in RAM!
scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(X_train_tfidf)
X_val_scaled = scaler.transform(X_val_tfidf)
print(f"-> Scaling completato in {time.time()-t0_scale:.1f}s\n")

# ---------------------------------------------------------------------------
# 3. ADDESTRAMENTO LOGISTIC REGRESSION (SOLVER='SAG', BALANCED)
# ---------------------------------------------------------------------------
t0_train = time.time()
print("3. Addestramento Logistic Regression (solver='sag', class_weight='balanced')...")

clf = LogisticRegression(
    solver='sag',
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1  # sfrutta tutti i core del cluster per velocizzare l'addestramento
)

clf.fit(X_train_scaled, y_train)
print(f"-> Addestramento completato in {time.time()-t0_train:.1f}s\n")

# ---------------------------------------------------------------------------
# 4. VALUTAZIONE SU VALIDATION SET
# ---------------------------------------------------------------------------
print("4. Predizione e calcolo metriche su Validation Set...")
y_pred = clf.predict(X_val_scaled)

macro_f1 = f1_score(y_val, y_pred, average='macro')
weighted_f1 = f1_score(y_val, y_pred, average='weighted')

print("=" * 60)
print(f" RISULTATI BASELINE 1 (LOGISTIC REGRESSION SAG + TF-IDF)")
print("=" * 60)
print(f"-> MACRO F1-SCORE:    {macro_f1:.4f}  (metrica ufficiale di riferimento)")
print(f"-> WEIGHTED F1-SCORE: {weighted_f1:.4f}")
print("=" * 60)

print("\n--- CLASSIFICATION REPORT DETTAGLIATO PER DIALETTO ---")
print(classification_report(y_val, y_pred, digits=4))

# ---------------------------------------------------------------------------
# 5. CONFUSION MATRIX PLOT
# ---------------------------------------------------------------------------
labels = sorted(list(set(y_val)))
cm = confusion_matrix(y_val, y_pred, labels=labels, normalize='true')

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues', ax=ax, values_format='.2f', xticks_rotation=45)
plt.title("Confusion Matrix Normalizzata - Baseline Logistic Regression (VarDial)")
plt.tight_layout()
plt.show()
